# 🌌 One2lv OS — Google Colab APK Compiler
**Target firmware:** `Android_FOXXD_A67M_Lite_V1.0_20250701`  
**App ID:** `com.one2lv.os`  
**Framework:** Capacitor 8.5 + Android SDK API 34  

### Steps
1. Install JDK 17, Node.js 20, Android SDK
2. Clone `one2lv-com/One2lvos` from GitHub
3. Install Capacitor dependencies
4. Sync Android platform
5. Build release APK with Gradle
6. Download the APK

> **Runtime:** Make sure Colab is set to a **CPU** or **T4 GPU** runtime (not TPU). Disk ~4 GB needed.

In [ ]:
# ============================================================
# CELL 1 — System environment: JDK 17 + Node.js 20
# ============================================================
import subprocess, os, sys

def run(cmd, **kw):
    """Run a shell command; print output; raise on failure."""
    print(f'▶ {cmd}')
    result = subprocess.run(cmd, shell=True, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kw)
    if result.stdout:
        print(result.stdout[-2000:])  # last 2000 chars to stay readable
    if result.returncode != 0:
        raise RuntimeError(f'Command failed (exit {result.returncode}): {cmd}')
    return result.stdout

print('⚡ Installing system dependencies...')
run('apt-get update -qq')
run('apt-get install -y -qq openjdk-17-jdk wget unzip curl')

# Install Node.js 20 via NodeSource
run('curl -fsSL https://deb.nodesource.com/setup_20.x | bash - 2>/dev/null')
run('apt-get install -y -qq nodejs')

# Set JAVA_HOME to JDK 17
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

print('\n✅ Versions:')
run('java -version 2>&1 || true')
run('node --version')
run('npm --version')

In [ ]:
# ============================================================
# CELL 2 — Android SDK command-line tools
# ============================================================
import os

SDK_ROOT  = '/content/android-sdk'
SDK_ZIP   = '/tmp/cmdtools.zip'
SDKTOOLS  = 'https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip'
SDKMGR    = f'{SDK_ROOT}/cmdline-tools/latest/bin/sdkmanager'

os.environ['ANDROID_HOME']     = SDK_ROOT
os.environ['ANDROID_SDK_ROOT'] = SDK_ROOT
os.makedirs(f'{SDK_ROOT}/cmdline-tools', exist_ok=True)

if not os.path.exists(SDKMGR):
    print('📥 Downloading Android SDK command-line tools...')
    run(f'wget -q {SDKTOOLS} -O {SDK_ZIP}')
    run(f'unzip -q -o {SDK_ZIP} -d {SDK_ROOT}/cmdline-tools')
    # The zip extracts to 'cmdline-tools/', rename to 'latest'
    extracted = f'{SDK_ROOT}/cmdline-tools/cmdline-tools'
    if os.path.isdir(extracted):
        os.rename(extracted, f'{SDK_ROOT}/cmdline-tools/latest')
    print('✅ SDK tools installed')
else:
    print('✅ SDK tools already present')

# Add sdk paths to PATH
os.environ['PATH'] = (
    f'{SDK_ROOT}/cmdline-tools/latest/bin:'
    f'{SDK_ROOT}/platform-tools:'
    + os.environ['PATH']
)

print('\n📦 Accepting licenses and installing API 34 + build-tools...')
run(f'yes | {SDKMGR} --licenses > /dev/null 2>&1 || true')
run(f'{SDKMGR} "platform-tools" "platforms;android-34" "build-tools;34.0.0"')
print('\n✅ Android SDK ready')

In [ ]:
# ============================================================
# CELL 3 — Clone One2lvOS repo (sparse: android project only)
# ============================================================
import os

REPO_URL   = 'https://github.com/one2lv-com/One2lvos.git'
CLONE_DIR  = '/content/One2lvos'
PROJ_DIR   = '/content/One2lv_Android'   # working build directory

if not os.path.isdir(CLONE_DIR):
    print('📥 Cloning One2lvOS repository (sparse)...')
    run(f'git clone --depth=1 --filter=blob:none --sparse {REPO_URL} {CLONE_DIR}')
    run(f'git -C {CLONE_DIR} sparse-checkout set one2lv-os-android')
else:
    print('📥 Updating existing clone...')
    run(f'git -C {CLONE_DIR} pull --depth=1')

# Copy android project to working directory
run(f'cp -r {CLONE_DIR}/one2lv-os-android {PROJ_DIR}')
print(f'\n✅ Project at {PROJ_DIR}')
run(f'ls {PROJ_DIR}')

In [ ]:
# ============================================================
# CELL 4 — Install Node dependencies + Capacitor CLI
# ============================================================
import os

PROJ_DIR = '/content/One2lv_Android'
os.chdir(PROJ_DIR)

print('📦 Installing npm packages...')
run('npm install --legacy-peer-deps')

print('\n📦 Installing Capacitor CLI globally...')
run('npm install -g @capacitor/cli@8')

print('\n✅ Dependencies installed')
run('npx cap --version')

In [ ]:
# ============================================================
# CELL 5 — Add Android platform and sync
# ============================================================
import os

PROJ_DIR = '/content/One2lv_Android'
os.chdir(PROJ_DIR)

os.environ['ANDROID_HOME']     = '/content/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/content/android-sdk'
os.environ['JAVA_HOME']        = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = (
    '/content/android-sdk/cmdline-tools/latest/bin:'
    '/content/android-sdk/platform-tools:'
    + os.environ['JAVA_HOME'] + '/bin:'
    + os.environ['PATH']
)

android_dir = os.path.join(PROJ_DIR, 'android')
if not os.path.isdir(android_dir):
    print('📱 Adding Android platform...')
    run('npx cap add android')
else:
    print('📱 Android platform already added')

print('\n🔄 Syncing web assets to Android...')
run('npx cap sync android')
print('\n✅ Android platform ready')

In [ ]:
# ============================================================
# CELL 6 — Configure Gradle for FOXXD target
# ============================================================
import os

PROJ_DIR  = '/content/One2lv_Android'
ANDROID   = os.path.join(PROJ_DIR, 'android')
APP_BUILD = os.path.join(ANDROID, 'app', 'build.gradle')

# Patch build.gradle: compileSdk 34, targetSdk 34, minSdk 24
with open(APP_BUILD, 'r') as f:
    gradle = f.read()

import re
gradle = re.sub(r'compileSdk\s+\d+',  'compileSdk 34',  gradle)
gradle = re.sub(r'targetSdk\s+\d+',   'targetSdk 34',   gradle)
gradle = re.sub(r'minSdk\s+\d+',      'minSdk 24',      gradle)
gradle = re.sub(r'versionName\s+"[^"]+"', 'versionName "1.0.20250701"', gradle)

with open(APP_BUILD, 'w') as f:
    f.write(gradle)

print('✅ build.gradle patched (API 34, minSdk 24)')

# Write local.properties so Gradle can find the SDK
local_props = os.path.join(ANDROID, 'local.properties')
with open(local_props, 'w') as f:
    f.write('sdk.dir=/content/android-sdk\n')
    f.write('ndk.dir=/content/android-sdk/ndk\n')

print('✅ local.properties written')

# Make gradlew executable
run(f'chmod +x {ANDROID}/gradlew')
print('✅ gradlew ready')

In [ ]:
# ============================================================
# CELL 7 — Build release APK
# ============================================================
import os, glob

PROJ_DIR = '/content/One2lv_Android'
ANDROID  = os.path.join(PROJ_DIR, 'android')
os.chdir(ANDROID)

os.environ['ANDROID_HOME']     = '/content/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/content/android-sdk'
os.environ['JAVA_HOME']        = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['GRADLE_OPTS']      = '-Xmx2g -XX:MaxMetaspaceSize=512m'
os.environ['PATH'] = (
    '/content/android-sdk/cmdline-tools/latest/bin:'
    '/content/android-sdk/platform-tools:'
    + os.environ['JAVA_HOME'] + '/bin:'
    + os.environ['PATH']
)

print('🔨 Building release APK (this takes 3–6 minutes)...')
print('   Target: Android_FOXXD_A67M_Lite_V1.0_20250701')
run('./gradlew assembleRelease --no-daemon --stacktrace 2>&1')

# Locate the APK
apks = glob.glob(f'{ANDROID}/**/app-release*.apk', recursive=True)
if apks:
    apk = apks[0]
    # Rename to match firmware target
    final = '/content/Android_FOXXD_A67M_Lite_V1.0_20250701.apk'
    os.rename(apk, final)
    size_mb = os.path.getsize(final) / (1024 * 1024)
    print(f'\n🎉 APK built successfully!')
    print(f'   Path: {final}')
    print(f'   Size: {size_mb:.1f} MB')
else:
    print('⚠️  APK not found — check Gradle output above for errors')

In [ ]:
# ============================================================
# CELL 8 — Download the APK to your computer
# ============================================================
import os
from google.colab import files

APK = '/content/Android_FOXXD_A67M_Lite_V1.0_20250701.apk'

if os.path.exists(APK):
    size_mb = os.path.getsize(APK) / (1024 * 1024)
    print(f'📲 Downloading APK ({size_mb:.1f} MB)...')
    files.download(APK)
    print('✅ Download triggered — check your browser downloads.')
else:
    print('❌ APK not found. Re-run Cell 7.')

In [ ]:
# ============================================================
# CELL 9 (optional) — Debug info: gradle/SDK versions
# ============================================================
import os

ANDROID = '/content/One2lv_Android/android'
os.chdir(ANDROID)

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['ANDROID_HOME'] = '/content/android-sdk'

print('--- Java ---')
run('java -version 2>&1 || true')
print('--- Gradle ---')
run('./gradlew --version --no-daemon 2>&1 | head -20')
print('--- SDK packages ---')
run('/content/android-sdk/cmdline-tools/latest/bin/sdkmanager --list_installed 2>/dev/null | head -20')

## Notes

| Item | Value |
|------|-------|
| App ID | `com.one2lv.os` |
| App Name | `One2lv OS` |
| Build target | `Android_FOXXD_A67M_Lite_V1.0_20250701` |
| Compile / Target SDK | API 34 |
| Min SDK | API 24 (Android 7.0+) |
| Build tools | 34.0.0 |
| JDK | 17 |
| Node | 20 |
| Capacitor | 8.5 |

### Installing on the FOXXD A67M Lite
1. Enable **Settings → Developer options → Unknown sources** (or USB debugging for ADB install)
2. Transfer `Android_FOXXD_A67M_Lite_V1.0_20250701.apk` to the device
3. Tap the file to install, or:
   ```bash
   adb install Android_FOXXD_A67M_Lite_V1.0_20250701.apk
   ```

### Signing (optional — for Play Store)
To sign with a release keystore, add this to Cell 7 before `assembleRelease`:
```
storeFile=<keystore path>
storePassword=<pw>
keyAlias=<alias>
keyPassword=<pw>
```
in `android/app/keystore.properties`, then reference it in `build.gradle`.